In [1]:
#importing necessary libraries
import pandas as pd
import numpy as np
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
from catboost import CatBoostRegressor
from sklearn.preprocessing import StandardScaler

In [2]:
 #loading dataset
df = pd.read_csv("../Data/TrialFinal.csv")

In [3]:
df.shape

(115, 12)

In [4]:
#null/missing values (show from eda that you identified this -999)
df['irradiance_SW_DWN']= df['irradiance_SW_DWN'].replace(-999, np.nan)   #sentinal value
df.isnull().sum()

year                   0
month                  0
month_num              0
wet_days               6
rainfall               6
male_workforce       105
female_workforce     105
temperature_T2M        0
irradiance_SW_DWN      1
EVI                   30
NDVI                  30
yield                 19
dtype: int64

In [5]:
#handling missing values in labour data
df['male_workforce']= (df.groupby('year')['male_workforce'].transform(lambda x: x.fillna(x.dropna().iloc[0])))
df['female_workforce']= (df.groupby('year')['female_workforce'].transform(lambda x: x.fillna(x.dropna().iloc[0])))

#handling correlation between male and female workforce

#creatine a total workforce column
df['total_workforce']=df['female_workforce']+df['male_workforce']

#creating female workforce as a ratio
df['female_workforceRatio']=df['female_workforce']/df['total_workforce']

df.drop(['male_workforce','female_workforce'], axis=1, inplace=True)

# Quick verification
print(df[['total_workforce', 'female_workforceRatio']].head())


   total_workforce  female_workforceRatio
0            901.0               0.560488
1            901.0               0.560488
2            901.0               0.560488
3            901.0               0.560488
4            901.0               0.560488


In [6]:
##new 15.02.2026- adding the code to show seasonality
df["sin_month"] = np.sin(2 * np.pi * df["month_num"] / 12)
df["cos_month"] = np.cos(2 * np.pi * df["month_num"] / 12)



In [7]:
df = df.dropna(subset=['yield']).reset_index(drop=True)

In [8]:
#adding lags
df = df.sort_values(['year', 'month_num']).reset_index(drop=True)
df['yield_lag_1']=df['yield'].shift(1)
df['yield_lag_2']=df['yield'].shift(2)
df['yield_lag_3']=df['yield'].shift(3)

#remove rows with missing lags
df=df.dropna(subset=['yield_lag_1', 'yield_lag_2', 'yield_lag_3']).reset_index(drop=True)

In [9]:
df = df.drop(columns=["month_num"])

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   year                   93 non-null     int64  
 1   month                  93 non-null     object 
 2   wet_days               89 non-null     float64
 3   rainfall               89 non-null     float64
 4   temperature_T2M        93 non-null     float64
 5   irradiance_SW_DWN      93 non-null     float64
 6   EVI                    78 non-null     float64
 7   NDVI                   78 non-null     float64
 8   yield                  93 non-null     float64
 9   total_workforce        93 non-null     float64
 10  female_workforceRatio  93 non-null     float64
 11  sin_month              93 non-null     float64
 12  cos_month              93 non-null     float64
 13  yield_lag_1            93 non-null     float64
 14  yield_lag_2            93 non-null     float64
 15  yield_la

In [11]:
df.shape

(93, 16)